# VisualForge AI on a free Kaggle GPU

Turns a narration script into a finished explainer video (with thumbnail and YouTube description), using the same
pipeline as the studio on your laptop. The GPU runs the language model, so **every scene gets AI-planned visuals**
(the laptop plans only the weakest scenes of long scripts).

**Before running:** in the right panel set *Accelerator* to **GPU T4 x2** (or P100), and turn *Internet* **on**
(Kaggle asks for phone verification once). Free accounts get about 30 GPU hours per week.

**Steps:** paste your script in the next cell, then *Run All*. Setup takes roughly 5–10 minutes; the video follows.
The results appear in `/kaggle/working/output` (download them from the *Output* panel).

In [ ]:
# Your narration script. Separate teaching points with blank lines (Markdown notes are converted automatically).
SCRIPT = """
Embeddings turn meaning into numbers that AI systems can compare. Let's see how text becomes tokens, how embeddings place meaning in space, and how retrieval uses them.

Think of embeddings as points in a huge mathematical space. If two concepts are similar in meaning, their vectors tend to be closer together. "Cat" and "Dog" may appear close together. "King" and "Queen" may also be close. But "Banana" may be much farther away from "Database."

So embeddings allow AI systems to represent meaning as geometry, and retrieval uses that geometry to find the right information for every question.
"""
TITLE = ""            # optional; by default the video is named from the script
VOICE = "af_heart"    # Kokoro voice id
MUSIC = False         # quiet background music bed under the narration
QUALITY = "final"     # "final" = 1080p, "draft" = 720p

In [ ]:
# Download the app into temporary storage (only the finished video is kept as notebook output).
!rm -rf /tmp/vf && git clone --depth 1 https://github.com/testvishal03/VisualForge-AI.git /tmp/vf
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# Install everything: system libraries, Node.js, Python 3.12 packages, renderer, models and the CUDA build of llama.cpp.
# If the CUDA build cannot start on this GPU, fall back to the CPU build so the video still gets made.
import subprocess
cuda = subprocess.run(['bash', '/tmp/vf/server/install_linux.sh', '--cuda']).returncode == 0
if not cuda:
    print('CUDA runtime could not start here; installing the CPU build instead.')
    subprocess.run(['bash', '/tmp/vf/server/install_linux.sh'], check=True)
print('Language model on', 'GPU' if cuda else 'CPU')

In [ ]:
# Make the video. On the GPU every scene is AI-planned; progress prints as each stage runs.
import os, pathlib
pathlib.Path('/tmp/script.txt').write_text(SCRIPT.strip(), encoding='utf-8')
env = {**os.environ, 'VISUALFORGE_GPU_LAYERS': '99' if cuda else '0', 'VISUALFORGE_PLANNING_LIMIT': '999' if cuda else '16'}
args = ['/tmp/vf/backend/.venv/bin/python', '/tmp/vf/backend/scripts/make_video.py', '--script', '/tmp/script.txt',
        '--out', '/kaggle/working/output', '--store', '/tmp/vf-store', '--voice', VOICE, '--quality', QUALITY]
args += (['--title', TITLE] if TITLE else []) + (['--music'] if MUSIC else [])
subprocess.run(args, check=True, env=env, cwd='/tmp/vf')

In [ ]:
# Watch the result here, then download video.mp4, thumbnail.png and description.txt from the Output panel.
from IPython.display import Video, Image, display
display(Video('/kaggle/working/output/video.mp4', embed=True, width=720))
display(Image('/kaggle/working/output/thumbnail.png', width=480))
print(open('/kaggle/working/output/description.txt', encoding='utf-8').read())